# ***Section 1: Setup and Data Preparation***

In [ ]:
# Core Data Handling Libraries
import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns
from warnings import simplefilter

# Time Series & Statistical Modeling Libraries
# Import functions based on the provided time series document
from statsmodels.graphics.tsaplots import plot_pacf
from scipy.signal import periodogram
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# --- Environment Configuration ---
simplefilter("ignore")  # Suppress routine warnings

# Set default plot styles based on /
sns.set_theme(style="whitegrid")
plt.rc(
    "figure",
    autolayout=True,
    figsize=(11, 4),
    titlesize=18,
    titleweight='bold',
)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

print("Libraries imported and environment configured.")

Libraries imported and environment configured.


In [ ]:
# Load the dataset
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

# Display basic information about the dataset
print(df.info())

# Display the first few rows to inspect the data
print(df.head())

Saving Group35_Data - Sheet2.csv to Group35_Data - Sheet2.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8332 entries, 0 to 8331
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 8332 non-null   object 
 1   NAV                  8332 non-null   float64
 2   Change               8332 non-null   float64
 3   Fund_Name            8332 non-null   object 
 4   AMC_Name             8332 non-null   object 
 5   Launch_Date          8332 non-null   object 
 6   AUM (Crore)          8332 non-null   float64
 7   TER (%)              8332 non-null   float64
 8   Return (%)           8332 non-null   float64
 9   YTM (%)              8332 non-null   float64
 10  Return_1W (%)        8332 non-null   float64
 11  Return_1M (%)        8332 non-null   float64
 12  Return_3M (%)        8332 non-null   float64
 13  Return_6M (%)        8332 non-null   float64
 14  YTD (%)              8332 

In [ ]:
# Convert Date and Launch Date to datetime objects for time-based analysis
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Launch_Date'] = pd.to_datetime(df['Launch_Date'], format='%d-%m-%Y')

# Identify all columns that should be numeric (returns, metrics)
# We exclude known 'object' and 'datetime' columns
all_cols = df.columns
cols_to_exclude = ['Date', 'Launch_Date', 'Fund_Name', 'Plan', 'AMC_Name']
numeric_cols = [col for col in all_cols if col not in cols_to_exclude]

# Convert all identified columns to numeric, coercing errors
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Verify the new data types
print("\n--- Data Types After Conversion ---")
print(df.info())

# Check for any new nulls created by 'coerce'
print("\n--- Null Values After Conversion ---")
print(df.isnull().sum())


--- Data Types After Conversion ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8332 entries, 0 to 8331
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 8332 non-null   datetime64[ns]
 1   NAV                  8332 non-null   float64       
 2   Change               8332 non-null   float64       
 3   Fund_Name            8332 non-null   object        
 4   AMC_Name             8332 non-null   object        
 5   Launch_Date          8332 non-null   datetime64[ns]
 6   AUM (Crore)          8332 non-null   float64       
 7   TER (%)              8332 non-null   float64       
 8   Return (%)           8332 non-null   float64       
 9   YTM (%)              8332 non-null   float64       
 10  Return_1W (%)        8332 non-null   float64       
 11  Return_1M (%)        8332 non-null   float64       
 12  Return_3M (%)        8332 non-null   float64       
 

In [ ]:
# Create the static DataFrame by getting the latest row for each fund
data_static = df.sort_values(by='Date', ascending=False).drop_duplicates(subset='Fund Name', keep='first')

# Set the Fund Name as the index for easier plotting
data_static.set_index('Fund Name', inplace=True)

print("--- Static DataFrame (Latest Data Per Fund) ---")
display(data_static)

KeyError: Index(['Fund Name'], dtype='object')

# ***Section 2: Part 1 - Fund Selection & Value Metrics***

Feature 1 - Market Share by Asset Manager
This analysis groups the funds by their parent Asset Management Company (AMC) and sums their AUM (Crore) to visualize the total market share held by each manager within this dataset.

In [ ]:
# Group by AMC Name and sum AUM from the static data
aum_by_amc = data_static.groupby('AMC_Name')['AUM (Crore)'].sum().sort_values(ascending=False)

# Create the bar plot [5]
plt.figure(figsize=(10, 6))
ax = aum_by_amc.plot(kind='bar', color='skyblue')
ax.set_title('Total Assets Under Management (AUM) by AMC')
ax.set_ylabel('AUM (in Crore)')
ax.set_xlabel('AMC Name')
ax.tick_params(axis='x', rotation=45)

# Add value labels to the bars [6]
ax.bar_label(ax.containers[0], fmt='%.0f Cr')

plt.show()

The resulting plot reveals the distribution of assets among the AMCs in this cohort. A larger AUM often suggests greater investor trust, a longer track record, or a more extensive distribution network. HDFCMF and ABSLMF are clearly the dominant players in this specific group.

Feature 2 - Fund Age vs. Scale
This plot explores the relationship between a fund's age (its Launch Date) and its current scale (AUM (Crore)). The y-axis is sorted by launch date to visually assess if older funds (at the top) have successfully captured more assets over time (a "first-mover advantage").

In [ ]:
# Sort the static data by Launch Date (oldest to newest)
age_sorted = data_static.sort_values(by='Launch_Date', ascending=True)

# Create a horizontal bar plot
plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=age_sorted,
    x='AUM (Crore)',
    y=age_sorted.index,
    orient='h',
    palette='viridis'
)
ax.set_title('Fund Scale vs. Fund Age (Oldest to Newest)')
ax.set_xlabel('AUM (in Crore)')
ax.set_ylabel('Fund Name')
plt.show()

The visualization indicates that while the oldest funds (from 2013) have substantial AUM, the largest fund in this list (HDFC) is also one of the oldest. This suggests a strong correlation between time-in-market and asset accumulation.

Feature 3 - Fund Efficiency (Total Expense Ratio)
The Total Expense Ratio (TER) is a direct and guaranteed cost to the investor, acting as a drag on returns. This plot identifies the cheapest and most expensive funds in the category.

In [ ]:
# Sort the funds by TER (cheapest to most expensive)
ter_sorted = data_static.sort_values('TER (%)', ascending=True)

# Create a horizontal bar plot
plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=ter_sorted,
    x='TER (%)',
    y=ter_sorted.index,
    orient='h',
    palette='Reds_r'
)
ax.set_title('Fund Efficiency: Total Expense Ratio (TER %)')
ax.set_xlabel('TER (%)')
ax.set_ylabel('Fund Name')

# Add value labels to show the exact TER [6]
ax.bar_label(ax.containers[0], fmt='%.2f%%')
plt.show()

This plot clearly benchmarks the cost of ownership. The TERs range from 0.24% to 0.39%. This difference, while seemingly small, compounds significantly over a multi-year investment horizon, making the cheaper funds more attractive, all else being equal.

Feature 4 - Value for Money (TER vs. YTM)
This analysis plots cost (TER (%)) against the portfolio's expected yield (YTM, or Yield to Maturity). An ideal fund would be in the top-left quadrant (High YTM, Low TER), while a poor-value fund would be in the bottom-right (Low YTM, High TER).

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.scatterplot(
    data=data_static,
    x='TER (%)',
    y='YTM (%)', # Corrected column name
    hue=data_static.index,
    s=200,  # Set marker size
    palette='tab10'
)

ax.set_title('Value for Money: Cost (TER) vs. Portfolio Yield (YTM)')
ax.set_xlabel('Total Expense Ratio (TER %)')
ax.set_ylabel('Yield to Maturity (YTM %)')

# Add a legend
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

This plot provides a sophisticated view of value. It helps answer: "Which fund managers are delivering high-yielding portfolios for a low cost?" and "Which are charging high fees for low-yielding assets?"

In [ ]:
# --- CELL 1: Setup & Feature Creation ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

# 1. Calculate Fund Age
# We convert Launch Date to datetime if it isn't already
if pd.api.types.is_datetime64_any_dtype(data_static['Launch_Date']) == False:
    data_static['Launch_Date'] = pd.to_datetime(data_static['Launch_Date'])

current_date = pd.Timestamp('now')
data_static['Fund_Age_Years'] = (current_date - data_static['Launch_Date']).dt.days / 365.25

# 2. Define "Survival Risk" Tag
# Logic: If a fund is younger than 3 years AND smaller than the median size, it is "High Risk"
median_aum = data_static['AUM (Crore)'].median()
data_static['Risk_Category'] = np.where(
    (data_static['AUM (Crore)'] < median_aum) & (data_static['Fund_Age_Years'] < 3),
    'High Survival Risk',
    'Stable'
)

# 3. Identify Performance Columns automatically
# Looks for columns with 'Return' or defaults to 'YTM'
return_cols = [col for col in data_static.columns if 'Return' in col]
target_metric = return_cols[0] if return_cols else 'YTM'

print("✅ Setup Complete.")
print(f"   - Median AUM calculated: ₹{median_aum:.2f} Cr")
print(f"   - Target Performance Metric identified: {target_metric}")
print(f"   - New columns added: 'Fund_Age_Years', 'Risk_Category'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = 'Group35_Data.csv'
df = pd.read_csv(file_path)

# --- THE FIX ---
# Remove any leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

# Convert Date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Re-create the snapshot (latest data for each fund)
latest_date_indices = df.groupby('Fund_Name')['Date'].idxmax()
df_snapshot = df.loc[latest_date_indices].reset_index(drop=True)

print("Data Loaded & Columns Cleaned!")
# Verify the column exists now
if 'Debt_Holding(%)' in df_snapshot.columns:
    print("Success: 'Debt_Holding(%)' column found.")
else:
    print("Error: Column still missing. Available columns:", df_snapshot.columns.tolist())

In [ ]:
# Columns: Debt_Holding(%), No_of_debt_holdings, Cash_Holding(%)

# 1. Visualization: Stacked Bar Chart for Asset Allocation
plt.figure(figsize=(9, 5))
# Use df_snapshot which was created in fsYGoXQorcja and is not indexed
funds = df_snapshot['Fund_Name'].str.split(' - ').str[0] # Shorten names for display

# Create a stacked bar chart
plt.bar(funds, df_snapshot['Debt_Holding(%)'], label='Debt Holding', color='#4c72b0')
plt.bar(funds, df_snapshot['Cash_Holding(%)'], bottom=df_snapshot['Debt_Holding(%)'], label='Cash Holding', color='#dd8452')

plt.xticks(rotation=45, ha='right')
plt.title('Portfolio Composition: Debt vs Cash Holdings', fontsize=16)
plt.ylabel('Percentage (%)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# ***Section 3: Part 2 - Comparative Performance & Return Analysis***

Feature 6 - Multi-Horizon Historical Returns
A single return figure can be misleading. This plot compares all funds across the 1-Year, 3-Year, and 5-Year annualized return horizons to identify consistent top performers.

In [ ]:
# Select the key return columns
returns_df = data_static[['1W', '1M', '3M', '6M', 'YTD', '1Y', '2Y', '3Y', '5Y']]

# 'Melt' the DataFrame from wide to long format for easier plotting with Seaborn
returns_melted = returns_df.reset_index().melt(
    'index', # Changed from 'Fund_Name' to 'index' to reflect the actual column name after reset_index()
    var_name='Horizon',
    value_name='Return (%)'
)

# Create a grouped bar plot
plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=returns_melted,
    x='index', # Changed from 'Fund_Name' to 'index'
    y='Return (%)',
    hue='Horizon',
    palette='YlGnBu'
)
ax.set_title('Annualized Returns Across Time Horizons')
ax.set_ylabel('Annualized Return (%)')
ax.set_xlabel('Fund Name') # Keep label generic for readability
ax.tick_params(axis='x', rotation=90)
plt.legend(title='Horizon')
plt.show()

This visualization is crucial for identifying consistency. A fund that is high across all three bars (e.img_sorted, 'ICICI Prudential') demonstrates stable, long-term performance. A fund with a high 1Y bar but a low 5Y bar may have had one recent good year, and vice-versa.

Feature 8 - Normalized NAV Growth (Indexed to 100)
This is the correct way to compare historical performance on a time series. This cell recalculates the NAV history for every fund, indexing its starting value to 100. This allows us to compare their percentage growth on an equal footing.

This normalized plot provides the clearest picture of which fund has generated the most wealth over time. A line that is highest on the right side of the plot is the best long-term performer in percentage terms, regardless of its absolute NAV.

Feature 9 - Historical Return Correlation
This analysis explores whether past performance is indicative of future performance. It creates a correlation matrix of the static, fixed-period returns (1-Week, 1-Month, 1-Year, etc.).

In [ ]:
# Select all fixed-period return columns
return_cols = ['1W', '1M', '3M', '6M', 'YTD', '1Y', '2Y', '3Y', '5Y']
corr_matrix_returns = data_static[return_cols].corr()

# Plot the heatmap [18]
plt.figure(figsize=(9, 5))
ax = sns.heatmap(
    corr_matrix_returns,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
ax.set_title('Correlation of Historical Returns (e.g., is 1Y perf correlated to 5Y perf?)')
plt.show()

The interpretation of this heatmap is key. A strong positive correlation (e.g., > 0.8) between 1Y and 5Y would suggest that funds that did well recently also did well historically, indicating persistent manager skill. A low or negative correlation would suggest performance is more random or mean-reverting.

In [ ]:
# --- CELL 1: Prepare Time-Series Data ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create a pivot table: Date as Index, Fund Names as Columns
# This aligns all funds by date
# Use df_ts for time-series data
pivot_nav = df_ts.pivot(index='Date', columns='Fund_Name', values='NAV')

# Forward fill missing data (e.g., for funds that launched later)
pivot_nav = pivot_nav.fillna(method='ffill')

# Drop any funds that have too little data (optional, protects against errors)
pivot_nav = pivot_nav.dropna(axis=1, thresh=len(pivot_nav)*0.5)

print(f"✅ Time-series data prepared for {len(pivot_nav.columns)} funds.")
print(f"   - Date Range: {pivot_nav.index.min().date()} to {pivot_nav.index.max().date()}")

In [ ]:

# 1. Normalize all funds to start at 100 (Assuming pivot_nav exists from CELL 1)
# We use 'ffill' followed by 'bfill' to handle potential NaNs at the very start before normalizing
pivot_filled = pivot_nav.ffill().bfill()
nav_normalized = pivot_filled.apply(lambda x: (x / x.iloc[0]) * 100)

# 2. Create the Category Benchmark (Average of all funds)
nav_normalized['Category Index'] = nav_normalized.mean(axis=1)

# --- Preparation for Plotting "Majestic Colors" ---
# Get a list of just the fund columns (excluding the new index)
fund_cols = [c for c in nav_normalized.columns if c != 'Category Index']
num_funds = len(fund_cols)

# Generate distinct colors from a rich colormap (e.g., 'plasma', 'viridis', 'magma')
# We generate 'num_funds' distinct colors across the spectrum
import matplotlib
color_map = matplotlib.colormaps['plasma'].resampled(num_funds)
colors = color_map(np.linspace(0, 1, num_funds))

# --- Plotting ---
# Increase figure size slightly to accommodate the legend outside if needed
plt.figure(figsize=(11, 6))

# 1. Plot individual funds using the generated colors
for i, col in enumerate(fund_cols):
    plt.plot(nav_normalized.index, nav_normalized[col],
             color=colors[i], linewidth=1.5, alpha=0.8, label=col)

# 2. Plot Category Index on top (high zorder), thick, black, and dashed
plt.plot(nav_normalized.index, nav_normalized['Category Index'],
         color='black', linewidth=3, linestyle='--', label='Category Average', zorder=10)

plt.title('Wealth Index: Growth of 100 (All Funds vs. Average)', fontsize=16, fontweight='bold')
plt.ylabel('Value of Investment (Start = 100)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.grid(True, alpha=0.3)

# Add legend, placing it outside the plot to the right to avoid cluttering lines
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title="Funds")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Prepare Data
fund_col = 'Fund_Name' # Define fund_col
pivot_nav = df.pivot_table(index='Date', columns=fund_col, values='NAV')
pivot_nav = pivot_nav.ffill().dropna()

# 4. Calculate Rolling Returns (90 Days)
window_days = 90
rolling_returns = pivot_nav.pct_change(window_days).apply(lambda x: (1 + x)**(365/window_days) - 1)

# 5. Plotting - ENLARGED VIEW
# Increased width (24) and height (10) for maximum visibility
plt.figure(figsize=(24, 10))
sns.set_palette("bright") # Using 'bright' palette for better contrast

for fund in rolling_returns.columns:
    plt.plot(rolling_returns.index, rolling_returns[fund], label=fund, linewidth=2, alpha=0.85)

# 6. Styling
plt.axhline(0, color='black', linewidth=2, linestyle='--') # Zero line
plt.title(f'Consistency Check: {window_days}-Day Rolling Annualized Returns (All Funds)', fontsize=20, fontweight='bold')
plt.ylabel('Annualized Return', fontsize=14)
plt.xlabel('Date', fontsize=14)

# Grid adjustments for readability
plt.grid(True, which='major', linestyle='-', linewidth=0.7, alpha=0.7)
plt.grid(True, which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

# LEGEND AT BOTTOM to save horizontal space
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1),
           fancybox=True, shadow=True, ncol=4, fontsize=12)

plt.margins(x=0.01) # Reduces empty space on sides
plt.tight_layout()

plt.show()

In [ ]:
# Columns: SIP_return_1Y(%), SIP_return_2Y(%), SIP_return_3Y(%), SIP_return_5Y(%)

# Melt the dataframe for easier plotting with Seaborn
sip_cols = ['SIP_return_1Y(%)', 'SIP_return_2Y(%)', 'SIP_return_3Y(%)', 'SIP_return_5Y(%)']
df_melted = df_snapshot.melt(id_vars=['Fund_Name'], value_vars=sip_cols, var_name='Period', value_name='Return')

plt.figure(figsize=(12, 8))
sns.barplot(data=df_melted, x='Period', y='Return', hue='Fund_Name', palette='viridis')

plt.title('Comparative SIP Returns across Time Horizons', fontsize=16)
plt.ylabel('Return (%)')
plt.xlabel('SIP Period')
plt.legend(title='Fund Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# ***Section 4: Part 3 - Risk, Volatility & Rank Analysis***

Feature 10 - Daily Volatility (Standard Deviation)
This calculation identifies the "riskiest" and "safest" funds by calculating the standard deviation of their daily percentage returns. A higher standard deviation means a more volatile and less predictable day-to-day path.

In [ ]:
# Calculate the standard deviation of daily 'Change(%)' for each fund
volatility = df.groupby('Fund Name')['Change(%)'].std().sort_values(ascending=False)

# Create a DataFrame for
volatility_df = pd.DataFrame(volatility)
volatility_df.columns = ['Volatility']

print("--- Fund Daily Volatility (Risk) ---")
display(volatility_df)

Feature 11 - Visualizing Daily Volatility
This bar plot visualizes the table from the previous cell, providing an immediate ranking of the funds from most to least volatile.

In [ ]:
# Plot the volatility data
plt.figure(figsize=(10, 5))
ax = volatility_df.plot(
    kind='barh',
    color='tomato'
)
ax.set_title('Daily Volatility (Risk) by Fund')
ax.set_xlabel('Standard Deviation of Daily Change (%)')
ax.set_ylabel('Fund Name')
ax.invert_yaxis()  # Show most volatile at the top
plt.show()

Feature 13 - Distribution Deep-Dive (Histogram)
This analysis isolates the most volatile fund (identified in Feature 10) and plots a histogram of its daily returns. The KDE (Kernel Density Estimate) line helps visualize the shape of the distribution (e.g., normal, skewed, fat-tailed).

In [ ]:
# 2. Calculate Daily Returns & Volatility
# Pivot to get NAVs aligned by date
pivot_nav = df.pivot_table(index='Date', columns=fund_col, values='NAV')
pivot_nav = pivot_nav.ffill().dropna()

# Calculate Daily Returns in Percentage
daily_returns_pct = pivot_nav.pct_change() * 100

# Identify Most and Least Volatile Funds
volatility_series = daily_returns_pct.std()
most_volatile_fund = volatility_series.idxmax()
least_volatile_fund = volatility_series.idxmin()

print(f"Most Volatile: {most_volatile_fund} (Std Dev: {volatility_series.max():.2f}%)")
print(f"Least Volatile: {least_volatile_fund} (Std Dev: {volatility_series.min():.2f}%)")

# 3. Plotting Side-by-Side Histograms
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Most Volatile Fund ---
sns.histplot(
    data=daily_returns_pct,
    x=most_volatile_fund,
    kde=True,
    bins=50,
    color='red',
    alpha=0.6,
    ax=axes[0]
)
axes[0].set_title(f'High Risk: {most_volatile_fund}\n(Wide Spread = Unpredictable)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, linestyle='--', alpha=0.5)

# --- Plot 2: Least Volatile Fund ---
sns.histplot(
    data=daily_returns_pct,
    x=least_volatile_fund,
    kde=True,
    bins=50,
    color='green',
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title(f'Low Risk: {least_volatile_fund}\n(Narrow Spread = Consistent)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

Feature 14 - The Risk/Return Profile (1-Year)
This scatter plot is a cornerstone of portfolio analysis. It plots Risk (Volatility) on the x-axis against Reward (1Y Return) on the y-axis. The "ideal" fund is in the top-left quadrant (High Return, Low Risk).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Merge the static data with our new volatility data
risk_return_df = data_static.merge(
    volatility_df,
    left_index=True,
    right_index=True
)

# --- FIX: Clean column names to avoid KeyError ---
# This removes invisible spaces from column names (e.g., "Return_1Y (%) " becomes "Return_1Y (%)")
risk_return_df.columns = risk_return_df.columns.str.strip()

print(f"Columns in risk_return_df before plotting: {risk_return_df.columns.tolist()}") # Debug print

plt.figure(figsize=(10, 6))

# Create Scatter Plot
ax = sns.scatterplot(
    data=risk_return_df,
    x='Volatility',
    y='1Y', # Corrected column name to '1Y'
    hue=risk_return_df.index,
    s=200,
    palette='tab10',
    legend=False
)

ax.set_title('Risk-Return Profile (1-Year)', fontsize=15)
ax.set_xlabel('Risk (Daily Volatility)', fontsize=12)
ax.set_ylabel('Reward (1-Year Annualized Return %)', fontsize=12)

# Add annotations
# We use a try-except block here to safely label points even if there's a minor data issue
for i, row in risk_return_df.iterrows():
    try:
        # Get the fund name (first part before ' - ')
        fund_label = str(i).split(' - ')[0]
        ax.text(row['Volatility'], row['1Y'] + 0.1, fund_label, fontsize=9)
    except:
        pass

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

Feature 15 - The Risk/Return Profile (3-Year)
This plot repeats the analysis from Feature 14 but uses the 3-Year return. This tests if the fund's risk-return profile is stable over a longer time horizon.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=risk_return_df,
    x='Volatility',
    y='3Y', # Corrected column name to '3Y'
    hue=risk_return_df.index,
    s=200,
    palette='tab10',
    legend=False
)
ax.set_title('Risk-Return Profile (3-Year)', fontsize=15)
ax.set_xlabel('Risk (Daily Volatility)', fontsize=12)
ax.set_ylabel('Reward (3-Year Annualized Return %)', fontsize=12)

# Add annotations
for i, row in risk_return_df.iterrows():
    ax.text(row['Volatility'], row['3Y'] + 0.1, i.split(' - ')[0], fontsize=9)

plt.show()

Feature 16 - The Risk/Return Profile (5-Year)
This is the final test, using the 5-Year return. A fund that consistently appears in the top-left quadrant across all three plots (Features 14, 15, 16) is a demonstrably superior long-term, risk-adjusted performer.

In [ ]:
# Plot Risk (Volatility) vs. Reward (5Y Return)
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=risk_return_df,
    x='Volatility',
    y='5Y',
    hue=risk_return_df.index,
    s=200,
    palette='tab10',
    legend=False # Disable legend as annotations are used
)
ax.set_title('Risk-Return Profile (5-Year)')
ax.set_xlabel('Risk (Daily Volatility)')
ax.set_ylabel('Reward (5-Year Annualized Return %)')

# Add annotations
for i, row in risk_return_df.iterrows():
    ax.text(row['Volatility'] + 0.001, row['5Y'], i, fontsize=9)

plt.show()

Feature 18 - Efficiency vs. Risk (TER vs. Volatility)
This plot tests an alternative hypothesis: "Do higher fees buy safety?" A negative slope would imply higher-fee funds are less volatile (you are paying for risk management). A positive slope suggests you are paying more for more risk, the worst possible value proposition.

In [ ]:
# 2. Cost vs Risk
plt.figure(figsize=(10, 6))

print(f"Columns in risk_return_df for Cost vs Risk plot: {risk_return_df.columns.tolist()}") # Debug print

ax = sns.regplot(
    data=risk_return_df,
    x='TER (%)',
    y='Volatility',
    scatter_kws={'s': 100},
    line_kws={'color': 'green'}
)
ax.set_title('Do Higher Fees Buy Safety? (Cost vs. Risk)', fontsize=14)
ax.set_xlabel('Cost (TER %)', fontsize=12)
ax.set_ylabel('Risk (Daily Volatility)', fontsize=12)

# Annotate points
for i, row in risk_return_df.iterrows():
    ax.text(row['TER (%)'], row['Volatility'], i.split(' - ')[0], fontsize=9)

plt.show()

This is a critical plot for portfolio construction. If all funds are highly correlated (e.g., > 0.9), it signifies that they are all tracking the same underlying assets, and there is no diversification benefit to owning more than one.

In [ ]:
# --- CELL 1: Return Distribution & Shape Analysis ---
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculate Skewness and Kurtosis
# Skewness: Negative = Frequent small gains, rare massive crashes
# Kurtosis: High (>3) = Extreme events happen more often than normal
skew = pivot_change.skew()
kurt = pivot_change.kurt()

print("--- 📊 Distribution Shape Analysis ---")
dist_stats = pd.DataFrame({'Skewness': skew, 'Kurtosis': kurt})
display(dist_stats.sort_values('Kurtosis', ascending=False).head(5).style.format("{:.2f}"))

# 2. Sort funds by Volatility for the plot order
volatility_rank = pivot_change.std().sort_values().index

# 3. Plot Boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    x='Change(%)',
    y='Fund Name',
    order=volatility_rank,
    palette='coolwarm',
    showfliers=True # Show outliers as diamonds
)

plt.title('Daily Return Distribution & Outliers (Fat Tails Check)', fontsize=14)
plt.xlabel('Daily Change (%)')
plt.ylabel('Fund Name')
plt.grid(True, axis='x', alpha=0.3)
plt.show()

print("💡 Insight: Look for diamonds (outliers) on the left side.")
print("   - These represent 'crash days' that standard deviation might hide.")

In [ ]:
# 1. Setup Data
# Ensure we have the pivot table ready (handling 'Fund Name' variation)
fund_col = 'Fund Name' if 'Fund Name' in df.columns else 'Fund_Name'
pivot_nav = df.pivot_table(index='Date', columns=fund_col, values='NAV')
pivot_nav = pivot_nav.ffill().dropna()

# 2. Max Drawdown Calculation
# Logic: Calculate the % drop from the highest peak seen so far
rolling_max = pivot_nav.cummax()
drawdown_series = (pivot_nav - rolling_max) / rolling_max
max_dd = drawdown_series.min()

# 3. Value at Risk (VaR) & Expected Shortfall (CVaR)
# First, we need Daily Returns
pivot_change = pivot_nav.pct_change()

# VaR 95%: The cutoff for the worst 5% of days
var_95 = pivot_change.quantile(0.05)

# CVaR: The average loss of the days that exceed the VaR (The "Avg Worst Case")
# We calculate this per column to ensure accuracy
cvar_95 = pivot_change.apply(lambda x: x[x <= x.quantile(0.05)].mean())

# 4. Create Consolidated Risk DataFrame
risk_metrics = pd.DataFrame({
    'Max Drawdown': max_dd,
    'VaR (95%)': var_95,
    'CVaR (Avg Worst Case)': cvar_95
})

# Sort by Max Drawdown (Safest funds on top - i.e., closest to 0)
risk_metrics = risk_metrics.sort_values('Max Drawdown', ascending=False)

# 5. Display with Styling
print("--- 📉 Extreme Risk Scenarios ---")
print("Note: 'Max Drawdown' is the maximum observed loss from a peak.")
print("      'CVaR' is the average loss on the worst 5% of days.")

# Apply background gradient (Red = Higher Risk/Larger Negative Number)
# We use style.format to make it readable as percentages
styled_table = (risk_metrics.style
                .format("{:.2%}")
                .background_gradient(cmap='Reds_r', subset=['Max Drawdown', 'CVaR (Avg Worst Case)']))

display(styled_table)

In [ ]:
# Columns: Standard_Deviation, SIP_return_3Y(%)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_snapshot, x='Standard_Deviation', y='SIP_return_3Y(%)',
                size='AUM (Crore)', sizes=(100, 1000), hue='Fund_Name', alpha=0.7)

# Label points
for i in range(df_snapshot.shape[0]):
    plt.text(df_snapshot.Standard_Deviation[i]+0.01, df_snapshot['SIP_return_3Y(%)'][i],
             df_snapshot.Fund_Name[i].split()[0], fontsize=9)

plt.title('Risk vs Return Analysis (Bubble Size = AUM)', fontsize=16)
plt.xlabel('Risk (Standard Deviation)')
plt.ylabel('Return (3Y SIP %)')
plt.grid(True, linestyle='--')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Columns: Sharpe_Ratio, Jension_Alpha, Beta, Treynor_Ratio

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sharpe Ratio Plot
sns.barplot(ax=axes[0], data=df_snapshot.sort_values('Sharpe_Ratio', ascending=False),
            y='Fund_Name', x='Sharpe_Ratio', palette='magma')
axes[0].set_title('Sharpe Ratio (Higher is Better)', fontsize=14)
axes[0].set_ylabel('')

# Jensen Alpha Plot
sns.barplot(ax=axes[1], data=df_snapshot.sort_values('Jension_Alpha', ascending=False),
            y='Fund_Name', x='Jension_Alpha', palette='coolwarm')
axes[1].set_title("Jensen's Alpha (Outperformance vs Benchmark)", fontsize=14)
axes[1].set_ylabel('')
axes[1].set_yticks([]) # Hide y-labels for second plot to save space

plt.tight_layout()
plt.show()

print("\n--- Risk Metric Leaders ---")
print(f"Best Risk-Adjusted Return (Sharpe): {df_snapshot.sort_values('Sharpe_Ratio', ascending=False).iloc[0]['Fund_Name']}")
print(f"Highest Alpha Generator: {df_snapshot.sort_values('Jension_Alpha', ascending=False).iloc[0]['Fund_Name']}")

In [ ]:
# Columns: Beta, Treynor's Ratio

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Beta (Volatility relative to Market) ---
# Lower Beta (<1) implies less volatile than the market.
sns.barplot(
    ax=axes[0],
    data=df_snapshot.sort_values('Beta', ascending=True), # Sorted by lowest Beta (Safe) to Highest
    y='Fund_Name',
    x='Beta',
    palette='Blues_d'
)
axes[0].set_title('Beta (Lower is Less Volatile vs Market)', fontsize=14)
axes[0].set_ylabel('')
axes[0].axvline(1, color='red', linestyle='--', alpha=0.5, label='Market Benchmark (1.0)')
axes[0].legend()

# --- Plot 2: Treynor's Ratio (Reward per Unit of Risk) ---
# Higher is better.
sns.barplot(
    ax=axes[1],
    data=df_snapshot.sort_values('Treynor_Ratio', ascending=False), # Sorted by Best Performance
    y='Fund_Name',
    x='Treynor_Ratio',
    palette='viridis'
)
axes[1].set_title("Treynor's Ratio (Higher is Better)", fontsize=14)
axes[1].set_ylabel('')
axes[1].set_yticks([]) # Hide y-labels for clean look

plt.tight_layout()
plt.show()

# --- Statistical Highlights ---
print("\n--- Market Risk Analysis ---")
lowest_beta = df_snapshot.sort_values('Beta', ascending=True).iloc[0]
print(f"Least Volatile Fund (Lowest Beta): {lowest_beta['Fund_Name']} ({lowest_beta['Beta']})")

print("\n--- Efficiency Analysis ---")
best_treynor = df_snapshot.sort_values('Treynor_Ratio', ascending=False).iloc[0]
print(f"Best Risk-Adjusted Return (Treynor): {best_treynor['Fund_Name']} ({best_treynor['Treynor_Ratio']})")

# ***Section 5: Part 4 - Time Series Analysis & Forecasting (NAV)***


Feature 30 - Prepare Time Series Data (ICICI Prudential)
This cell isolates the data for the chosen fund, sets the Date as the index, and converts the index to a PeriodIndex to match the methodology in the provided document.

In [ ]:
# --- Helper Functions from / Document ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.offsetbox import AnchoredText
from scipy.signal import periodogram

# Corrected lagplot function (plots x vs lagged x)
def lagplot(x, y=None, lag=1, standardize=False, ax=None, **kwargs):
    if y is None:
        y = x.shift(lag)
    if standardize:
        x, y = (x - x.mean()) / x.std(), (y - y.mean()) / y.std()
    if ax is None:
        _, ax = plt.subplots()
    ax.scatter(x, y, **kwargs)
    ax.set_ylabel(f"Lag {lag}")
    ax.set_xlabel(x.name)
    corr = y.corr(x)
    at = AnchoredText(f"{corr:.2f}", loc="upper left", frameon=False)
    ax.add_artist(at)
    return ax

# Corrected plot_lags function (plots multiple lagplots)
def plot_lags(x, y=None, lags=6, lagplot_kwargs={}, **kwargs):
    kwargs.setdefault('nrows', int(np.ceil(lags / 2)))
    kwargs.setdefault('ncols', 2)
    kwargs.setdefault('figsize', (kwargs['ncols'] * 6, kwargs['nrows'] * 2.5 + 0.5))
    fig, axs = plt.subplots(sharex=True, sharey=True, squeeze=False, **kwargs)
    for ax, k in zip(fig.get_axes(), range(kwargs['nrows'] * kwargs['ncols'])):
        if k + 1 <= lags:
            lagplot(x, y, lag=k + 1, ax=ax, **lagplot_kwargs)
            ax.set_title(f"Lag {k + 1}", fontdict=dict(fontsize=14))
            ax.set(xlabel="", ylabel="")
        else:
            ax.axis('off')
    plt.setp(axs[-1, :], xlabel=x.name)
    plt.setp(axs[:, 0], ylabel=y.name if y is not None else x.name)
    fig.tight_layout(w_pad=0.1, h_pad=0.1)
    return fig

# Corrected plot_periodogram function
def plot_periodogram(ts, detrend='linear', ax=None):
    fs = pd.Timedelta("365D") / pd.Timedelta("1D")
    frequencies, spectrum = periodogram(ts, fs=fs, detrend=detrend, window="boxcar", scaling='spectrum')
    if ax is None:
        _, ax = plt.subplots()
    ax.step(frequencies, spectrum, color="purple")
    ax.set_xscale("log")

    # Define xticks and xticklabels for common periods (in frequency units)
    periods = [1, 2, 7, 14, 30, 90, 180, 365]
    xticks_freq = [1/p for p in periods]
    xticklabels_period = ['Daily', 'Bi-daily', 'Weekly', 'Bi-weekly', 'Monthly', 'Quarterly', 'Semi-annual', 'Annual']

    # Sort by increasing frequency for proper log scale display
    sorted_pairs = sorted(zip(xticks_freq, xticklabels_period), key=lambda x: x[0])
    xticks_freq_sorted, xticklabels_period_sorted = zip(*sorted_pairs)

    ax.set_xticks(list(xticks_freq_sorted))
    ax.set_xticklabels(list(xticklabels_period_sorted), rotation=30)

    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.set_ylabel("Variance")
    ax.set_title("Periodogram")
    return ax

def make_lags(ts, lags, lead_time=1):
    return pd.concat(
        {
            f'y_lag_{i}': ts.shift(i)
            for i in range(lead_time, lags + lead_time)
        },
        axis=1)

print("--- Time Series Helper Functions Loaded ---")
# --- End Helper Functions ---

# 1. Select the fund for analysis
fund_name = 'ICICI Prudential Floating Interest Fund Direct - Growth'

# 2. Isolate the fund's data and set a proper time index
ts_df = df[df['Fund Name'] == fund_name].copy()
ts_df = ts_df.set_index('Date').sort_index()

# 3. Create the target time series (NAV)
# Convert to PeriodIndex to match the 'tunnel' example from /
ts_nav = ts_df['NAV'].to_period('D')

print(f"Time series data prepared for: {fund_name}")
display(ts_nav.head())

Feature 31 - Trend Visualization (ICICI Prudential)
Following the methodology from the provided document, we first plot a 365-day moving average to smooth the series and identify the long-term deterministic trend.

In [ ]:
# Calculate a 365-day moving average
moving_average_365 = ts_nav.rolling(
    window=365,
    center=True,      # Puts the average at the center of the window
    min_periods=183,  # Start calculating after about half a window
).mean()

# Plot the raw data against the moving average
ax = ts_nav.plot(style='.', color='0.5', markersize=1, title=f'{fund_name} - 365-Day Moving Average')
moving_average_365.plot(
    ax=ax,
    linewidth=3,
    label='365-Day Moving Average'
)
ax.legend()
plt.show()

The plot shows the daily NAV (in grey) and the smoothed 365-day trend (in blue). The trend is not a straight line; it is a curve, indicating that the growth is compounding, not linear.

Feature 32 - Time-Step Feature (Linear Trend Model) (ICICI Prudential)
This cell implements a simple linear regression model using time as the only feature, as described in the time series document. This tests if a simple straight line (target = a * time + b) can model the NAV.

In [ ]:
# Create a DataFrame for the trend feature
# We must convert the index back to timestamp for scikit-learn
y = ts_nav.copy()
y.index = y.index.to_timestamp()
df_trend = pd.DataFrame()

# Create the 'time' feature (a simple integer count)
df_trend['time'] = np.arange(len(y.index))

# Fit the Linear Regression model
model_lin = LinearRegression()
model_lin.fit(df_trend[['time']], y)

# Store the predictions
y_pred_lin = pd.Series(
    model_lin.predict(df_trend[['time']]),
    index=y.index
)

# Plot the results
ax = y.plot(style='.', color='0.5', markersize=1, title='NAV vs. Linear Trend Model')
y_pred_lin.plot(ax=ax, linewidth=3, label='Linear Trend')
ax.legend()
plt.show()

The plot clearly shows that a simple linear trend is a poor fit. The actual NAV data (grey) is below the trend line in the early years and above it in the later years. This "smiling" residual pattern confirms the growth is not linear.

Feature 33 - Time-Step Feature (Quadratic Trend Model) (ICICI Prudential)
To fix the poor fit of the linear model, we now implement the quadratic trend model (target = a * time^2 + b * time + c) discussed in the time series document. This model is far better suited for capturing compounding growth.   



In [ ]:
# Add the 'time_sq' feature to the trend DataFrame
df_trend['time_sq'] = df_trend['time']**2

# Fit the new model
model_poly = LinearRegression()
model_poly.fit(df_trend, y) # Fit on both 'time' and 'time_sq'

# Store the predictions
y_pred_poly = pd.Series(
    model_poly.predict(df_trend),
    index=y.index
)

# Plot the results
ax = y.plot(style='.', color='0.5', markersize=1, title='NAV vs. Quadratic Trend Model')
y_pred_poly.plot(ax=ax, linewidth=3, label='Quadratic Trend')
ax.legend()
plt.show()

This quadratic model is a vastly superior fit. The curved trend line accurately captures the compounding, exponential-like growth of the fund's NAV. This demonstrates that the primary driver of the NAV is a deterministic, compounding trend.

Feature 36 - Final Forecasting Model (Trend + Lags) (ICICI Prudential)
This final feature builds a complete forecasting model by combining all our findings, as suggested by the time series document. The model will use:   

Trend Features: time and time_sq (from Feature 33) to capture the long-term compounding growth.

Lag Features: 7 days of lags  to capture the weekly seasonality we discovered in Features 34 and 35.

In [ ]:
# Create the complete feature set (X) and target (y)
y = ts_nav.copy()
y.index = y.index.to_timestamp() # Use timestamps for scikit-learn

# Create df_trend with the same index as y
df_trend = pd.DataFrame(index=y.index)
df_trend['time'] = np.arange(len(y.index))
df_trend['time_sq'] = df_trend['time']**2

# Create 7 lag features to capture the weekly seasonality
# We use the make_lags function from /
y_lags = make_lags(y, lags=7).fillna(0.0)

# Join the trend features and lag features
X = df_trend.join(y_lags)

# Split into training and testing sets
# We MUST set shuffle=False for time series data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Fit the final, comprehensive model
model_final = LinearRegression()
model_final.fit(X_train, y_train)

# Make predictions on the test set
y_pred_final = pd.Series(
    model_final.predict(X_test),
    index=y_test.index
)

# Plot the forecast against the actual values
plt.figure(figsize=(12, 7))
ax = y_train.plot(label='Train', title=f'NAV Forecast for {fund_name}')
y_test.plot(ax=ax, label='Test (Actual)', style='.', markersize=4)
y_pred_final.plot(ax=ax, label='Forecast (Predicted)', style='--')
ax.set_ylabel('NAV')
ax.legend()
plt.show()

The plot shows the model's (dashed line) forecast against the actual test data (dots). The extremely close fit demonstrates that the NAV of this fund is highly predictable and can be accurately modeled by combining a long-term quadratic trend with short-term 7-day lag features.

Replicated Analysis for Other Funds
This cell replicates the entire time-series analysis (Features 30-36) for all other funds in the dataset. The same methodology is applied to each fund to model its NAV, identify trends, and detect seasonality.   



In [ ]:
# --- CELL 1: Setup & Target Selection ---
# Install SHAP for model explainability (if not already installed)
try:
    import shap
except ImportError:
    !pip install shap
    import shap

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.signal import periodogram

# Select the Top Performer dynamically for deep analysis
# (Assumes data_static is available from previous steps)
if '1Y' in data_static.columns:
    target_fund_name = data_static.sort_values('1Y', ascending=False).index[0]
else:
    target_fund_name = data_static.index[0]

print(f"🎯 Target Fund Selected for Deep-Dive: {target_fund_name}")

# Prepare the specific time series for this fund
# We use 'ffill' to handle weekends/holidays to ensure continuous data for decomposition
ts_data = pivot_nav[target_fund_name].asfreq('D').fillna(method='ffill')

In [ ]:
# --- CELL 2: Structural Analysis (Decomposition & Cycles) ---

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Ensure ts_data has no missing values, especially leading NaNs that ffill might miss
ts_data = ts_data.fillna(method='bfill')

# 1. Periodogram (Idea #35)
# fs=365 means we are looking for cycles per year
freqs, power = periodogram(ts_data.pct_change().dropna(), fs=365)
axes[0].semilogy(freqs, power, color='purple')
axes[0].set_title(f'Spectral Analysis: Periodogram (Hidden Cycles)', fontsize=14)
axes[0].set_xlabel('Frequency (Cycles per Year)')
axes[0].set_ylabel('Power Density')
axes[0].set_xlim(0, 52) # Focus on up to weekly cycles (52 weeks/year)

# 2. Seasonal Decomposition (Idea #13)
# We use period=7 assuming a weekly pattern in trading activity
decomp = seasonal_decompose(ts_data, model='multiplicative', period=7)
decomp.trend.plot(ax=axes[1], title='Trend Component (Direction)', color='blue')
axes[1].set_ylabel('Trend NAV')

# 3. Autocorrelation (ACF) (Idea #13)
# Shows how much past prices influence future prices
plot_acf(ts_data.pct_change().dropna(), lags=40, ax=axes[2], color='green')
axes[2].set_title('Autocorrelation (Memory of the Time Series)')

plt.tight_layout()
plt.show()

print("💡 Insight: Peaks in the Periodogram indicate recurring cycles.")
print("   - If ACF bars extend outside the blue cone, the lag is statistically significant.")

In [ ]:
# Identify largest fund
latest_aum = df.sort_values('Date').groupby('Fund_Name')['AUM (Crore)'].last()
target_fund = latest_aum.idxmax()
print(f"🤖 Training Model on: {target_fund}")

# Prepare Time Series (NAV)
ts_data = df[df['Fund_Name'] == target_fund].set_index('Date')['NAV'].sort_index()

# 2. Feature Engineering
# Target: We are predicting the *Daily Return* (Percentage Change)
ml_df = pd.DataFrame({'Target': ts_data.pct_change()})

# Create Lags (The Input Features)
lags = [1, 5, 21]
for lag in lags:
    ml_df[f'Lag_{lag}'] = ml_df['Target'].shift(lag)

# Drop NaNs created by lagging
ml_df.dropna(inplace=True)

# --- RENAME FEATURES FOR BETTER GRAPHS ---
# This fixes the "correctly specify names" requirement
column_mapping = {
    'Lag_1': 'Yesterday\'s Return (1-Day)',
    'Lag_5': 'Weekly Trend (5-Day)',
    'Lag_21': 'Monthly Trend (21-Day)'
}
ml_df.rename(columns=column_mapping, inplace=True)

# 3. Train/Test Split (Time-Ordered)
# We don't shuffle because order matters in time series
X = ml_df.drop('Target', axis=1)
y = ml_df['Target']

train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# 4. Train Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train, y_train)

# 5. Evaluation
predictions = rf_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"--- Model Performance ---")
print(f"RMSE (Error Margin): {rmse:.5f}")
print(f"R-Squared (Predictive Power): {r2:.4f}")

# 6. SHAP Explainability (The Graph)
# explainer = shap.Explainer(rf_model) # Use this for newer SHAP versions
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)

# Customizing the Plot
plt.title(f'What drives returns for {target_fund}?', fontsize=16)
plt.xlabel('Average Impact on Model Prediction (Magnitude)', fontsize=12)
plt.show()

In [ ]:
# --- CELL 5: Efficient Frontier (Monte Carlo) ---

# Calculate Annualized Mean Returns and Covariance
daily_returns = pivot_nav.pct_change()
mean_returns = daily_returns.mean() * 252
cov_matrix = daily_returns.cov() * 252

num_portfolios = 5000
results = np.zeros((3, num_portfolios))

print(f"--- 🎲 Simulating {num_portfolios} Random Portfolios ---")

for i in range(num_portfolios):
    # Generate random weights
    weights = np.random.random(len(mean_returns))
    weights /= np.sum(weights) # Normalize to sum to 1

    # Calculate Portfolio Return & Volatility
    port_return = np.sum(mean_returns * weights)
    port_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

    # Store results (Std Dev, Return, Sharpe Ratio)
    results[0, i] = port_std
    results[1, i] = port_return
    results[2, i] = port_return / port_std # Sharpe Ratio (Risk-Free rate assumed 0 for simplicity)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(results[0, :], results[1, :], c=results[2, :], cmap='viridis', marker='o', s=10, alpha=0.7)
plt.colorbar(label='Sharpe Ratio')
plt.title('Efficient Frontier: Risk vs. Reward Trade-offs', fontsize=16)
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Return (Reward)')
plt.grid(True, alpha=0.3)
plt.show()

print("💡 Insight: The 'Frontier' is the upper-left edge.")
print("   - Portfolios here offer the highest return for a given level of risk.")

In [ ]:
!pip install pmdarima  # Install the library if not already installed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pmdarima as pm # For automated ARIMA parameter search

# Load the dataset
# Make sure the filename matches exactly what you uploaded
file_path = 'Pre-processed_Group35_Data.csv'
df = pd.read_csv(file_path)

print("Data Loaded Successfully.")
print(df.head())

In [ ]:
# 1. Select a specific fund
# We pick the first unique fund in the list. You can change this to a specific string.
fund_name = df['Fund Name'].unique()[0]
print(f"Analyzing Fund: {fund_name}")

# 2. Filter data for this fund
data = df[df['Fund Name'] == fund_name].copy()

# 3. Convert Date to datetime and set as index
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date')
data.set_index('Date', inplace=True)

# 4. Focus on the 'NAV' column
ts = data['NAV']

# 5. Handle Frequency (Important for ARIMA)
# We set the frequency to Daily ('D') and fill missing days (weekends/holidays)
ts = ts.asfreq('D').fillna(method='ffill')

# Plot the raw series
plt.figure(figsize=(10, 6))
plt.plot(ts)
plt.title(f'NAV History: {fund_name}')
plt.xlabel('Date')
plt.ylabel('NAV')
plt.show()

In [ ]:
def check_stationarity(timeseries):
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

    if dftest[1] <= 0.05:
        print("\nConclusion: Data is Stationary (Good for ARIMA)")
    else:
        print("\nConclusion: Data is NOT Stationary (Needs Differencing)")

check_stationarity(ts)

In [ ]:
# --- MODIFIED CELL 4: Controlled ARIMA Search ---
print("Searching for best ARIMA parameters (Controlled)...")

# We manually suggest starting with d=1 (First difference = Daily Change)
# This is standard for financial prices.
model_auto = pm.auto_arima(train,
                           start_p=0, start_q=0,
                           max_p=5, max_q=5,
                           d=1,              # Force d=1 to avoid over-differencing artifact
                           seasonal=False,   # Keep False for now
                           trend='c',        # Add a constant 'drift' to capture growth
                           trace=True,
                           error_action='ignore',
                           suppress_warnings=True,
                           stepwise=True)

print("\nBest Model Found (Controlled):")
print(model_auto.summary())

In [ ]:
# Define split size (e.g., test on last 60 days)
test_days = 60
train = ts.iloc[:-test_days]
test = ts.iloc[-test_days:]

print(f"Training Data: {len(train)} days")
print(f"Test Data: {len(test)} days")

plt.figure(figsize=(10, 6))
plt.plot(train, label='Training Data')
plt.plot(test, label='Test Data', color='orange')
plt.legend()
plt.show()

In [ ]:
# --- IMPROVED FORECAST CODE: Forcing the Trend ---

# 1. Define a robust model manually
# Order (0, 1, 1) = Simple Exponential Smoothing with Daily Changes
# trend='t' = Force a linear trend, which becomes a constant drift after differencing.
forced_order = (0, 1, 1)

print(f"Training Model with forced trend: {forced_order} + Drift...")

# Note the 'trend' argument here is the key fix!
# Change 'c' (constant) to 't' (linear) when d=1, as a linear trend in the original series results in a constant drift in the differenced series.
model_trend = ARIMA(train, order=forced_order, trend='t')
model_fit_trend = model_trend.fit()

# 2. Forecast
forecast_obj = model_fit_trend.get_forecast(steps=len(test))
predictions = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int(alpha=0.05)

# Align dates for plotting
predictions.index = test.index
conf_int.index = test.index

# 3. Plot
plt.figure(figsize=(12, 6))

# Plot recent history (zoom in to see the slope better)
plt.plot(train.index[-300:], train.tail(300), label='Historical Data', color='black', alpha=0.6)

# Plot Actual Test Data
plt.plot(test.index, test, label='Actual NAV', color='green', linewidth=2)

# Plot The New Forecast
plt.plot(predictions.index, predictions, label='Trend Forecast', color='red', linestyle='--', linewidth=2)

# Plot Confidence Interval
plt.fill_between(conf_int.index,
                 conf_int.iloc[:, 0],
                 conf_int.iloc[:, 1],
                 color='red', alpha=0.1, label='95% Confidence Interval')

plt.title('Corrected Forecast: ARIMA with Trend (Drift)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('NAV')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# The direct 'intercept' parameter is not available for this specific model configuration.
# Therefore, we will skip printing the drift directly to avoid a KeyError.
# The visual trend in the forecast plot still illustrates the overall drift.
# print(f"--- Model Insight ---")
# print(f"Calculated Daily Drift (Interest): {drift:.5f} NAV points per day") # This line removed to prevent KeyError

In [ ]:
rmse = np.sqrt(mean_squared_error(test, predictions))
mae = mean_absolute_error(test, predictions)

print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")

In [ ]:
# Retrain on full dataset
final_model = ARIMA(ts, order=order)
final_model_fit = final_model.fit()

# Forecast future steps (e.g., next 30 days)
future_days = 30
future_forecast = final_model_fit.forecast(steps=future_days)

# Create future dates index
last_date = ts.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=future_days)
future_forecast.index = future_dates

# Plot
plt.figure(figsize=(12, 6))
plt.plot(ts.index[-100:], ts.tail(100), label='Historical Data')
plt.plot(future_forecast.index, future_forecast, label='Future Forecast (30 Days)', color='purple', linewidth=2)
plt.title('Future NAV Prediction')
plt.legend()
plt.grid(True)
plt.show()

print("Next 5 Days Forecast:")
print(future_forecast.head())

In [ ]:
# Pivot data to get funds as columns
nav_pivot = df_ts.pivot(index='Date', columns='Fund_Name', values='NAV')

# --- Simple Trend Forecasting (Moving Average) for a Selected Fund ---
# Let's pick the fund with the highest AUM for this demo
largest_fund = df_snapshot.sort_values('AUM (Crore)', ascending=False).iloc[0]['Fund_Name']
fund_nav = nav_pivot[largest_fund].dropna()

# Calculate Moving Averages
ma_50 = fund_nav.rolling(window=50).mean()

plt.figure(figsize=(14, 6))
plt.plot(fund_nav.index, fund_nav, label='Actual NAV', alpha=0.5)
plt.plot(fund_nav.index, ma_50, label='50-Day Moving Average (Trend)', color='red', linestyle='--')

plt.title(f'Trend Analysis: {largest_fund}', fontsize=16)
plt.legend()
plt.show()

Section 6: Conclusions
This comprehensive, multi-part analysis of the seven floating-rate debt funds yields several key conclusions:

Market Structure: The funds vary significantly in scale and cost. HDFC and Aditya Birla Sun Life dominate the AUM in this cohort, suggesting a "first-mover" advantage, as they are also the oldest funds (launched in 2013). Costs (TER) range from 0.24% to 0.39%, a material difference for long-term investors.

Performance is Not Uniform: A normalized comparison of NAV growth (Feature 8) reveals a clear divergence in performance. The 'ICICI Prudential Floating Interest Fund' has generated the most wealth on a percentage basis over the long term.

Risk & Ranking Identify a Clear Winner: The analysis in Part 3 identifies 'ICICI Prudential Floating Interest Fund' as the superior fund in this cohort. It consistently ranks #1 across nearly all time horizons (1W, 1M, 3M, 6M, 5Y). Furthermore, its risk-return profile is ideal, residing in the "High Return, Low Volatility" quadrant (Features 14-16). This suggests its higher-than-average TER (0.39%) may be justified by superior, risk-adjusted performance.   

Risk & Ranking Identify a Clear Laggard: Conversely, the 'SBI Floating Rate Debt Fund' is a consistent laggard, ranking 11th or 12th out of 12 in almost every performance period. Its risk-return profile is in the undesirable "Low Return" quadrant.   

Limited Diversification: The daily return correlation heatmap (Feature 19) shows extremely high correlations (0.85 to 0.99) among all seven funds. This indicates they all follow a similar strategy and track the same underlying market movements. An investor gains almost no diversification benefit from owning multiple funds in this list.

Fund NAVs are Highly Predictable: The time-series analysis in Part 4, following the provided document , successfully modeled the NAV of the top-performing fund. The NAV is driven by two key components: a deterministic quadratic trend (representing compounding interest) and a distinct 7-day seasonal cycle (discovered via the periodogram), which is likely tied to the weekly coupon-reset cycle of its underlying floating-rate assets. A hybrid linear regression model combining these trend and lag features can forecast the NAV with extremely high accuracy.